# Analisi primaria Qwen2.5-0.5B su Tenstorrent Blackhole P150a

Questo notebook consolida le quattro campagne primarie di `tt-blackhole-benchmark` e analizza prestazioni, potenza ed energia per input 128/512, output 32/128 e batch 1/8/32.

Le finestre energetiche sono ricavate dai marker temporali di TT-Metal:

- **prefill**: `Starting prefill` → `Prefill finished`;
- **decode**: `Starting decode loop` → `Finished decoding`;
- **end-to-end**: inizio prefill → fine decode.

L'energia end-to-end rappresenta il comportamento osservato dello stack corrente e può includere compilazione/cattura della trace. Non è quindi automaticamente confrontabile con un engine persistente TensorRT-LLM o con il protocollo SweetSpot da 1000 richieste.


In [ ]:
from pathlib import Path
import re
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path("..")
RESULTS_ROOT = PROJECT_ROOT / "results"
EXPORT_ROOT = PROJECT_ROOT / "analysis" / "exports"
FIGURE_ROOT = PROJECT_ROOT / "analysis" / "figures"
LOG_TIMEZONE = "Europe/Rome"

assert RESULTS_ROOT.is_dir(), f"Directory risultati mancante: {RESULTS_ROOT}"
print("Project:", PROJECT_ROOT)
print("Python analysis ready")


## 1. Individuazione automatica delle campagne valide

Per ogni geometria del workload viene selezionata la campagna più recente contenente esattamente 12 run completati e nessun run fallito. In questo modo vengono ignorati automaticamente i tentativi incompleti o falliti.


In [ ]:
def run_statuses(database: Path) -> dict:
    with sqlite3.connect(database) as connection:
        return dict(connection.execute(
            "SELECT status, COUNT(*) FROM runs GROUP BY status"
        ).fetchall())


def telemetry_failure_count(database: Path) -> int:
    with sqlite3.connect(database) as connection:
        return connection.execute(
            "SELECT COUNT(*) FROM telemetry_failures"
        ).fetchone()[0]


def find_latest_valid_campaign(name_pattern: str) -> Path:
    candidates = sorted(
        RESULTS_ROOT.glob(f"*_{name_pattern}_*"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    diagnostics = []

    for root in candidates:
        database = root / "benchmark.sqlite"
        if not database.is_file():
            diagnostics.append((root.name, "database mancante"))
            continue

        statuses = run_statuses(database)
        failures = telemetry_failure_count(database)
        diagnostics.append((root.name, statuses, failures))

        if statuses == {"completed": 12} and failures == 0:
            return root

    raise RuntimeError(
        f"Nessuna campagna valida per {name_pattern}. Candidati: {diagnostics}"
    )


CAMPAIGNS = {
    "in128_out128": find_latest_valid_campaign("qwen-primary-in128-out128"),
    "in128_out32": find_latest_valid_campaign("qwen-primary-in128-out32"),
    "in512_out128": find_latest_valid_campaign("qwen-primary-in512-out128"),
    "in512_out32": find_latest_valid_campaign("qwen-primary-in512-out32"),
}

for name, root in CAMPAIGNS.items():
    print(f"{name:14} -> {root}")


## 2. Parsing delle metriche e dei marker TT-Metal

I log TT-Metal usano l'ora locale del server, mentre la telemetria è memorizzata in UTC. I timestamp vengono quindi localizzati in `Europe/Rome` e successivamente convertiti in UTC.


In [ ]:
SPEED_PATTERN = re.compile(
    r"Average speed: "
    r"(?P<latency_ms>[0-9.]+)ms @ "
    r"(?P<tokens_s_user>[0-9.]+) tok/s/user "
    r"\((?P<throughput_tokens_s>[0-9.]+) tok/s throughput\)"
)

TTFT_PATTERN = re.compile(r"Average Time to First Token \(TTFT\): (?P<ttft_ms>[0-9.]+)ms")
PREFILL_COMPILE_PATTERN = re.compile(r"Prefill compile time: (?P<seconds>[0-9.]+)s")
DECODE_COMPILE_PATTERN = re.compile(r"Decode compile time: (?P<seconds>[0-9.]+)s")
LOG_TIMESTAMP_PATTERN = re.compile(
    r"^(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d+)"
)

MARKERS = {
    "prefill_start": "test_demo_text:1180 - Starting prefill...",
    "prefill_end": "test_demo_text:1197 - Prefill finished",
    "decode_start": "test_demo_text:1226 - Starting decode loop...",
    "decode_trace_ready": "Done Capturing Decode Trace",
    "decode_end": "test_demo_text:1325 - Finished decoding",
}


def unique_float(pattern: re.Pattern, text: str, field: str, required=True):
    matches = list(pattern.finditer(text))
    if not matches and not required:
        return np.nan
    if len(matches) != 1:
        raise ValueError(f"Attesa una metrica {field}, trovate {len(matches)}")
    group = matches[0].groupdict()
    return float(group.get(field, group.get("seconds")))


def parse_log(stderr_path: Path) -> dict:
    text = stderr_path.read_text(encoding="utf-8", errors="replace")
    speed_matches = list(SPEED_PATTERN.finditer(text))
    if len(speed_matches) != 1:
        raise ValueError(f"{stderr_path}: trovate {len(speed_matches)} metriche Average speed")

    speed = {key: float(value) for key, value in speed_matches[0].groupdict().items()}
    speed["ttft_ms"] = unique_float(TTFT_PATTERN, text, "ttft_ms")
    speed["prefill_compile_s"] = unique_float(
        PREFILL_COMPILE_PATTERN, text, "seconds", required=False
    )
    speed["decode_compile_s"] = unique_float(
        DECODE_COMPILE_PATTERN, text, "seconds", required=False
    )

    markers = {}
    for line in text.splitlines():
        timestamp_match = LOG_TIMESTAMP_PATTERN.match(line)
        if timestamp_match is None:
            continue
        timestamp = (
            pd.Timestamp(timestamp_match.group("timestamp"))
            .tz_localize(LOG_TIMEZONE, ambiguous="raise", nonexistent="raise")
            .tz_convert("UTC")
        )
        for name, marker_text in MARKERS.items():
            if marker_text in line:
                markers[name] = timestamp

    missing = set(MARKERS) - set(markers)
    if missing:
        raise ValueError(f"{stderr_path}: marker mancanti {sorted(missing)}")

    return {**speed, **markers}


## 3. Caricamento e validazione

Vengono caricati i run e i campioni telemetrici di tutte le campagne. Le successive asserzioni bloccano l'analisi se il dataset non corrisponde alla matrice prevista.


In [ ]:
runs_parts = []
telemetry_parts = []
log_rows = []

for campaign_name, campaign_root in CAMPAIGNS.items():
    database = campaign_root / "benchmark.sqlite"
    with sqlite3.connect(database) as connection:
        campaign_runs = pd.read_sql_query(
            "SELECT * FROM runs ORDER BY rowid", connection
        )
        campaign_telemetry = pd.read_sql_query(
            "SELECT * FROM telemetry_samples ORDER BY run_id, monotonic_ns", connection
        )

    campaign_runs["campaign_name"] = campaign_name
    campaign_telemetry["campaign_name"] = campaign_name
    runs_parts.append(campaign_runs)
    telemetry_parts.append(campaign_telemetry)

    for _, run in campaign_runs.iterrows():
        parsed = parse_log(campaign_root / run["stderr_path"])
        log_rows.append({"run_id": run["run_id"], **parsed})

runs = pd.concat(runs_parts, ignore_index=True)
telemetry = pd.concat(telemetry_parts, ignore_index=True)
log_metrics = pd.DataFrame(log_rows)

telemetry["captured_at"] = pd.to_datetime(telemetry["captured_at"], utc=True)
runs = runs.merge(log_metrics, on="run_id", how="left", validate="one_to_one")

assert len(runs) == 48, f"Attesi 48 run, trovati {len(runs)}"
assert set(runs["status"]) == {"completed"}
assert runs["run_id"].is_unique
assert telemetry["run_id"].isin(runs["run_id"]).all()
assert set(runs["input_tokens_requested"]) == {128, 512}
assert set(runs["output_tokens_requested"]) == {32, 128}
assert set(runs["batch_size"]) == {1, 8, 32}
assert set(runs["run_phase"]) == {"warmup", "measurement"}
assert (runs.query("run_phase == 'measurement'").groupby(
    ["input_tokens_requested", "output_tokens_requested", "batch_size"]
).size() == 3).all()

print(f"Run validi: {len(runs)}")
print(f"Measurement: {(runs['run_phase'] == 'measurement').sum()}")
print(f"Campioni telemetrici: {len(telemetry):,}")


## 4. Integrazione della potenza

L'integrazione usa il metodo trapezoidale e interpola la potenza ai confini delle finestre. Si conservano separatamente energia di prefill, decode, decode dopo la cattura della trace ed end-to-end.


In [ ]:
def integrate_power_window(samples, start, end):
    samples = samples.sort_values("captured_at")
    surrounding = samples.loc[
        (samples["captured_at"] >= start - pd.Timedelta(seconds=0.2))
        & (samples["captured_at"] <= end + pd.Timedelta(seconds=0.2))
    ]
    if len(surrounding) < 2:
        available = (samples["captured_at"].min(), samples["captured_at"].max())
        raise ValueError(
            f"Campioni insufficienti per {start} -> {end}; disponibili {available}"
        )

    timestamp_ns = surrounding["captured_at"].astype("int64").to_numpy()
    power_w = surrounding["power_w"].to_numpy(dtype=float)
    start_ns, end_ns = start.value, end.value
    start_power = np.interp(start_ns, timestamp_ns, power_w)
    end_power = np.interp(end_ns, timestamp_ns, power_w)
    inside = (timestamp_ns > start_ns) & (timestamp_ns < end_ns)
    window_ns = np.concatenate(([start_ns], timestamp_ns[inside], [end_ns]))
    window_power = np.concatenate(([start_power], power_w[inside], [end_power]))
    time_s = (window_ns - start_ns) / 1e9
    duration_s = (end_ns - start_ns) / 1e9
    energy_j = np.trapezoid(window_power, time_s)
    return {
        "duration_s": duration_s,
        "energy_j": energy_j,
        "mean_power_w": energy_j / duration_s,
        "max_power_w": float(window_power.max()),
    }


telemetry_by_run = {
    run_id: group.copy() for run_id, group in telemetry.groupby("run_id")
}

energy_rows = []
for _, run in runs.iterrows():
    samples = telemetry_by_run[run["run_id"]]
    windows = {
        "prefill": (run["prefill_start"], run["prefill_end"]),
        "decode": (run["decode_start"], run["decode_end"]),
        "decode_steady": (run["decode_trace_ready"], run["decode_end"]),
        "inference": (run["prefill_start"], run["decode_end"]),
    }
    row = {"run_id": run["run_id"]}
    for prefix, (start, end) in windows.items():
        values = integrate_power_window(samples, start, end)
        row.update({f"{prefix}_{key}": value for key, value in values.items()})
    energy_rows.append(row)

window_energy = pd.DataFrame(energy_rows)
runs = runs.merge(window_energy, on="run_id", how="left", validate="one_to_one")

runs["output_tokens_total"] = (
    runs["request_count"] * runs["output_tokens_requested"]
)
runs["inference_joules_per_token"] = (
    runs["inference_energy_j"] / runs["output_tokens_total"]
)
runs["inference_tokens_per_joule"] = (
    runs["output_tokens_total"] / runs["inference_energy_j"]
)
runs["estimated_generation_s"] = (
    runs["output_tokens_total"] / runs["throughput_tokens_s"]
)
runs["generation_fraction_percent"] = (
    100 * runs["estimated_generation_s"] / runs["inference_duration_s"]
)

assert runs.filter(regex="_energy_j$").notna().all().all()
assert (runs.filter(regex="_energy_j$") > 0).all().all()
print("Integrazione completata per", len(runs), "run")


## 5. Dataset delle misurazioni e controlli di plausibilità

In [ ]:
measurement_runs = runs.loc[runs["run_phase"] == "measurement"].copy()
measurement_runs["workload"] = (
    measurement_runs["input_tokens_requested"].astype(str)
    + "/"
    + measurement_runs["output_tokens_requested"].astype(str)
)

diagnostic_columns = [
    "workload", "batch_size", "repetition_index",
    "throughput_tokens_s", "tokens_s_user", "ttft_ms",
    "prefill_duration_s", "decode_duration_s", "inference_duration_s",
    "inference_mean_power_w", "inference_energy_j",
    "inference_joules_per_token", "inference_tokens_per_joule",
    "generation_fraction_percent",
]

display(
    measurement_runs.sort_values(
        [
            "input_tokens_requested",
            "output_tokens_requested",
            "batch_size",
            "repetition_index",
        ]
    )[diagnostic_columns]
)

display(measurement_runs[[
    "inference_duration_s", "inference_mean_power_w", "inference_energy_j",
    "inference_joules_per_token", "inference_tokens_per_joule",
    "generation_fraction_percent",
]].describe())


## 6. Aggregazione statistica

Il coefficiente di variazione è riportato in percentuale. Le medie energetiche sono calcolate esclusivamente sulle tre measurement, escludendo i warm-up.


In [ ]:
GROUP_COLUMNS = [
    "input_tokens_requested", "output_tokens_requested", "batch_size"
]
METRICS = [
    "throughput_tokens_s", "tokens_s_user", "ttft_ms",
    "inference_duration_s", "inference_mean_power_w", "inference_energy_j",
    "inference_joules_per_token", "inference_tokens_per_joule",
]

summary = measurement_runs.groupby(GROUP_COLUMNS)[METRICS].agg(["mean", "std"])
summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary = summary.reset_index()

for metric in METRICS:
    summary[f"{metric}_cv_percent"] = (
        100 * summary[f"{metric}_std"] / summary[f"{metric}_mean"]
    )

summary["workload"] = (
    summary["input_tokens_requested"].astype(str)
    + "/"
    + summary["output_tokens_requested"].astype(str)
)

display(summary[[
    "workload", "batch_size",
    "throughput_tokens_s_mean", "throughput_tokens_s_std", "throughput_tokens_s_cv_percent",
    "inference_mean_power_w_mean", "inference_energy_j_mean",
    "inference_joules_per_token_mean", "inference_tokens_per_joule_mean",
]].sort_values(["workload", "batch_size"]))


## 7. Rilevamento robusto degli outlier di throughput

Con tre repliche non è opportuno eliminare automaticamente alcuna misura. Il modified z-score basato sulla mediana serve soltanto a identificare punti da ripetere.


In [ ]:
def modified_z_score(values):
    values = np.asarray(values, dtype=float)
    median = np.median(values)
    mad = np.median(np.abs(values - median))
    if mad == 0:
        return np.zeros_like(values)
    return 0.6745 * (values - median) / mad


measurement_runs["throughput_modified_z"] = (
    measurement_runs.groupby(GROUP_COLUMNS)["throughput_tokens_s"]
    .transform(lambda values: modified_z_score(values))
)
measurement_runs["throughput_outlier_candidate"] = (
    measurement_runs["throughput_modified_z"].abs() > 3.5
)

outliers = measurement_runs.loc[
    measurement_runs["throughput_outlier_candidate"],
    ["workload", "batch_size", "repetition_index", "throughput_tokens_s", "throughput_modified_z"],
]
display(outliers if not outliers.empty else pd.DataFrame({"result": ["Nessun candidato"]}))


## 8. Grafici prestazionali ed energetici

In [ ]:
plot_data = summary.sort_values(["workload", "batch_size"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.lineplot(
    data=plot_data, x="batch_size", y="throughput_tokens_s_mean",
    hue="workload", marker="o", ax=axes[0],
)
axes[0].set_title("Throughput aggregato")
axes[0].set_xlabel("Batch size")
axes[0].set_ylabel("Token/s")
axes[0].set_xticks([1, 8, 32])

sns.lineplot(
    data=plot_data, x="batch_size", y="inference_tokens_per_joule_mean",
    hue="workload", marker="o", ax=axes[1],
)
axes[1].set_title("Efficienza energetica end-to-end")
axes[1].set_xlabel("Batch size")
axes[1].set_ylabel("Token/J")
axes[1].set_xticks([1, 8, 32])

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.lineplot(
    data=plot_data, x="batch_size", y="inference_mean_power_w_mean",
    hue="workload", marker="o", ax=axes[0],
)
axes[0].set_title("Potenza media durante l'inferenza")
axes[0].set_xlabel("Batch size")
axes[0].set_ylabel("W")
axes[0].set_xticks([1, 8, 32])

sns.lineplot(
    data=plot_data, x="batch_size", y="inference_joules_per_token_mean",
    hue="workload", marker="o", ax=axes[1],
)
axes[1].set_title("Energia per token end-to-end")
axes[1].set_xlabel("Batch size")
axes[1].set_ylabel("J/token")
axes[1].set_yscale("log")
axes[1].set_xticks([1, 8, 32])

plt.tight_layout()
plt.show()


## 9. Profilo temporale di un run rappresentativo

Il grafico consente di verificare visivamente la corrispondenza tra marker e profilo di potenza.


In [ ]:
mask = (
    (measurement_runs["input_tokens_requested"] == 512)
    & (measurement_runs["output_tokens_requested"] == 128)
    & (measurement_runs["batch_size"] == 32)
    & (measurement_runs["repetition_index"] == 0)
)
example = measurement_runs.loc[mask].iloc[0]
samples = telemetry_by_run[example["run_id"]].sort_values("captured_at").copy()
samples["relative_s"] = (
    samples["captured_at"] - example["prefill_start"]
).dt.total_seconds()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(samples["relative_s"], samples["power_w"], linewidth=1.2, label="Power")

for column, label, color in [
    ("prefill_start", "Prefill start", "tab:green"),
    ("prefill_end", "Prefill end", "tab:olive"),
    ("decode_start", "Decode start", "tab:orange"),
    ("decode_trace_ready", "Trace ready", "tab:red"),
    ("decode_end", "Decode end", "tab:purple"),
]:
    relative = (example[column] - example["prefill_start"]).total_seconds()
    ax.axvline(relative, linestyle="--", color=color, label=label)

ax.set_xlim(-2, example["inference_duration_s"] + 2)
ax.set_xlabel("Tempo rispetto all'inizio del prefill [s]")
ax.set_ylabel("Potenza [W]")
ax.set_title("Profilo di potenza — input 512, output 128, batch 32")
ax.legend(ncol=3)
plt.tight_layout()
plt.show()


## 10. Esportazione di dati, tabelle e figure

Gli output derivati vengono salvati fuori da `results/`, che rimane immutata. Prima del commit verificare che la politica del repository per `analysis/exports` e `analysis/figures` sia quella desiderata.


In [ ]:
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

measurement_runs.to_csv(EXPORT_ROOT / "qwen_primary_measurements.csv", index=False)
summary.to_csv(EXPORT_ROOT / "qwen_primary_summary.csv", index=False)
outliers.to_csv(EXPORT_ROOT / "qwen_primary_outlier_candidates.csv", index=False)

latex_table = summary[[
    "workload", "batch_size", "throughput_tokens_s_mean", "throughput_tokens_s_std",
    "inference_mean_power_w_mean", "inference_energy_j_mean",
    "inference_joules_per_token_mean", "inference_tokens_per_joule_mean",
]].rename(columns={
    "workload": "Input/Output",
    "batch_size": "Batch",
    "throughput_tokens_s_mean": "Throughput medio [tok/s]",
    "throughput_tokens_s_std": "Dev. std. throughput",
    "inference_mean_power_w_mean": "Potenza media [W]",
    "inference_energy_j_mean": "Energia media [J]",
    "inference_joules_per_token_mean": "J/token",
    "inference_tokens_per_joule_mean": "token/J",
}).sort_values(["Input/Output", "Batch"])

latex_path = EXPORT_ROOT / "qwen_primary_summary.tex"
latex_path.write_text(
    latex_table.to_latex(index=False, float_format=lambda value: f"{value:.3f}"),
    encoding="utf-8",
)

print("Esportati:")
for path in sorted(EXPORT_ROOT.iterdir()):
    print("-", path)


## Interpretazione e limiti

- Il throughput riportato da TT-Metal è una metrica interna del decode e non coincide necessariamente con il tempo end-to-end osservato.
- `inference_tokens_per_joule` usa l'intervallo dal prefill effettivo alla fine del decode; include l'eventuale compilazione/cattura della trace interna al run.
- `decode_steady_energy_j` è utile per studiare la fase successiva alla cattura della trace, ma non viene normalizzata automaticamente su tutti i token perché una parte della generazione può avvenire durante la preparazione della trace.
- Le tre repliche consentono una prima stima della variabilità, ma i punti con CV elevato o modified z-score anomalo devono essere ripetuti.
- Il confronto diretto con SweetSpot richiede modelli comuni, un numero di richieste comparabile e una chiara armonizzazione del perimetro energetico.
